# Longitudinal trajectory classification — common notebook (adapter-driven)

One model-agnostic notebook for every longitudinal trajectory experiment. The
per-model logic lives in an **adapter** (`CLASSIFIER/adapters/`) selected by the
`ADAPTER` parameter (defaults to `MODEL`); the SHARED cells below call only the
six contract hooks plus `common/` utilities, so they are identical across models.

GELSTM / GEGRU and the GEC-MLP are covered by two adapters; FDR and GRU are config
flags (`use_fdr`, `rnn_type`), not separate notebooks. Driven by `run_experiment.py`
via `the experiments/ directory`; also runnable standalone (interactive prompts preserved).

In [1]:
# === Papermill parameters (injected by run_experiment.py) ===
# Safe interactive defaults: None keeps the original Jupyter behaviour
# (interactive checkpoint/threshold prompts).
EXPERIMENT_ID = None
MODE = None
MODEL = None
ADAPTER = None                # adapter registry key; None -> defaults to MODEL
DATASET = None
SEED = 42
GAAE_CHECKPOINT_PATH = None   # None -> interactive checkpoint picker
THRESHOLD_MODE = None         # None -> interactive prompt; else youden | best-f1 | fixed
FIXED_THRESHOLD = None        # required when THRESHOLD_MODE is fixed
WANDB_ENABLED = True          # W&B logging is on by default
OUTPUT_DIR = None             # defaults to a common-notebook checkpoints dir when standalone
RESOLVED_CONFIG = None        # merged hyperparameter dict (dataclass < json < hyperparams)
RUN_DIR = None                # set by the runner: where run_summary.json / artifacts go
RUN_NAME = None               # set by the runner: the W&B run name

In [2]:
# Parameters
EXPERIMENT_ID = "tfgn-s1c-recon-pooled-seed45"
MODE = "longitudinal"
MODEL = "TFGN"
DATASET = "POOLED_WHOLE_BRAIN"
SEED = 45
GAAE_CHECKPOINT_PATH = None
THRESHOLD_MODE = "best-f1"
FIXED_THRESHOLD = None
WANDB_ENABLED = True
OUTPUT_DIR = "outputs/tfgn-s1c-recon-pooled-seed45"
RESOLVED_CONFIG = {"epochs": 100, "lr": 0.001, "weight_decay": 0.0, "batch_size": 16, "grad_clip": 1.0, "early_stopping_patience": 20, "use_scheduler": True, "seed": 45, "lr_factor": 0.5, "lr_patience": 5, "lr_min": 1e-06, "n_rois": 200, "lstm_hidden": 64, "lstm_layers": 1, "lstm_dropout": 0.3, "use_time_delta": True, "gvae_hidden": 128, "gvae_latent": 64, "gvae_heads": 2, "gvae_dropout": 0.3, "adjacency_k": 8, "node_lstm_init": "pretrained_finetuned", "node_lstm_ckpt_path": "outputs/tfgn-nodelstm-ssl-pooled/runs/morning-violet-1-fa30969e6-2026-08-24_08-36-00/checkpoint_morning-violet-1-fa30969e6-2026-08-24_08-36-00.pth", "use_gate": False, "lambda_sparse": 0.1, "lambda_drift": 0.01, "gate_rho": 0.15, "recon_target": "delta_a_topk", "lambda_recon": 1.0, "beta_kl": 1.0, "free_bits": 0.5, "beta_warmup_epochs": 5.0, "change_mask_kappa": 0.1, "fusion": "z_only", "readout": "mean", "dual_score": False, "lambda_cent": 0.1, "tau": 0.05, "cohort_conditioning": "none", "encoder_init": "none", "gvae_ckpt_path": None, "zero_time_delta": False, "graph_pool": "mean", "dim_filter": None, "shuffle_order": False, "shuffle_rng": None, "threshold_mode": "youden", "fixed_threshold": 0.5, "encoder_grad": False, "external_test_cohort": "oasis3", "min_visits": 2, "defer_test_eval": True}
RUN_DIR = "/mnt/e/fyassine/ad-early-detection/CLASSIFIER/outputs/tfgn-s1c-recon-pooled-seed45/runs/kind-haze-3-d0d4f8a68-2026-08-24_13-55-50"
RUN_NAME = "kind-haze-3-d0d4f8a68-2026-08-24_13-55-50"
ADAPTER = "tfgn"


## Pipeline overview

`set_seed` → load splits → `prepare_data` (HOOK) → `build_model` (HOOK) →
`run_kfold_cv` (drives `train_fold` HOOK) → `select_oof_threshold` → `save_run` →
test (`eval_split` HOOK) → ROC → `early_detection_table` (`truncate_to_n_visits`) →
`trajectory_frame` (`per_visit_probs`).

In [3]:
import sys
from pathlib import Path
repo_root = Path('/mnt/e/fyassine/ad-early-detection')
model_root = Path('/mnt/e/fyassine/ad-early-detection/CLASSIFIER')
if str(model_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
    sys.path.insert(0, str(model_root))

In [4]:
# reproducibility seeding — must run before datasets, samplers, or models.
from SHARED.seeding import (
    set_seed, make_rng, make_torch_generator, seed_worker,
)
set_seed(SEED)
rng = make_rng(SEED)
torch_gen = make_torch_generator(SEED)

In [5]:
import json, os, copy, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from datetime import datetime
from sklearn.metrics import classification_report

# Shared, model-agnostic utilities (the lifted notebook logic — reuse, do not inline).
from common.checkpoints import select_gaae_checkpoint
from SHARED.sanity import run_full_audit
from SHARED import tracking
from SHARED.provenance import region_from_data_root
from common.crossval import Bundle, run_kfold_cv, summarize_cv
from common.thresholds import select_oof_threshold
from common.plots import plot_oof_test_roc, plot_conversion_trajectories
from common.early_detection import early_detection_table, trajectory_frame
from common.run_artifacts import save_run, record_test_metrics
from adapters import get_adapter

warnings.filterwarnings('ignore')
print('Imports OK')

Imports OK


In [6]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

Device: cuda


## Configuration

In [7]:
from DATA.DELCODE.src.splitting.load_splits import splits_dir, split_csv_paths

# ── Paths ────────────────────────────────────────────────────────────────
dataset_str = str(DATASET or '').upper()
if 'ADNI' in dataset_str:
    WB_DATA_ROOT = str(repo_root / 'DATA/ADNI/__fc_wholebrain_sch200_flat__/matrices')
    METADATA_DIR = str(repo_root / 'DATA/ADNI/__metadata__')
    COHORTS_CSV  = os.path.join(METADATA_DIR, 'cohort_manifest.csv')
    SPLITS_DIR   = os.path.join(METADATA_DIR, 'SPLITS', 'downstream')
    TRAIN_CSV    = os.path.join(SPLITS_DIR, 'train.csv')
    VAL_CSV      = os.path.join(SPLITS_DIR, 'val.csv')
    TEST_CSV     = os.path.join(SPLITS_DIR, 'test.csv')
    cohort_tag   = 'adni'
elif 'OASIS' in dataset_str:
    WB_DATA_ROOT = str(repo_root / 'DATA/OASIS3/__fc_wholebrain_sch200_flat__/matrices')
    METADATA_DIR = str(repo_root / 'DATA/OASIS3/__metadata__')
    COHORTS_CSV  = os.path.join(METADATA_DIR, 'cohort_manifest.csv')
    SPLITS_DIR   = os.path.join(METADATA_DIR, 'SPLITS', 'downstream')
    TRAIN_CSV    = os.path.join(SPLITS_DIR, 'train.csv')
    VAL_CSV      = os.path.join(SPLITS_DIR, 'val.csv')
    TEST_CSV     = os.path.join(SPLITS_DIR, 'test.csv')
    cohort_tag   = 'oasis3'
elif 'POOLED' in dataset_str:
    # Pooled ADNI+DELCODE — DOCS/flipped/PLAN.md Phase 1 / DATA/manifest/build_pooled_assets.py.
    # WB_DATA_ROOT points at the symlink farm purely so region_from_data_root's
    # naming parse still works; the actual per-cohort FC roots used at load time
    # come from CLASSIFIER.common.pooled_data.COHORT_ROOTS (see prepare_data HOOK).
    WB_DATA_ROOT = str(repo_root / 'DATA/POOLED_ADNI_DELCODE/__fc_wholebrain_sch200_flat__/matrices')
    METADATA_DIR = str(repo_root / 'DATA/POOLED_ADNI_DELCODE')
    COHORTS_CSV  = None  # unused for pooled frames; each cohort's dataset ignores it too
    SPLITS_DIR   = os.path.join(METADATA_DIR, 'SPLITS', 'downstream')
    TRAIN_CSV    = os.path.join(SPLITS_DIR, 'train.csv')
    VAL_CSV      = os.path.join(SPLITS_DIR, 'val.csv')
    TEST_CSV     = os.path.join(SPLITS_DIR, 'test.csv')
    cohort_tag   = 'pooled'
else:
    WB_DATA_ROOT = '/mnt/e/fyassine/ad-early-detection/DATA/DELCODE/__fc_wholebrain_sch200_flat__/matrices'
    METADATA_DIR = '/mnt/e/fyassine/ad-early-detection/DATA/DELCODE/__fc_wholebrain_sch200_flat__/metadata'
    COHORTS_CSV  = os.path.join(METADATA_DIR, 'cohorts_with_scans_on_disk.csv')
    SPLITS_DIR   = str(splits_dir('downstream'))
    TRAIN_CSV    = os.path.join(SPLITS_DIR, 'train.csv')
    VAL_CSV      = os.path.join(SPLITS_DIR, 'val.csv')
    TEST_CSV     = os.path.join(SPLITS_DIR, 'test.csv')
    cohort_tag   = 'delcode'

DATA_INFO = region_from_data_root(WB_DATA_ROOT)
REGION    = DATA_INFO['region']
print(f"Input data: cohort={cohort_tag} region={DATA_INFO['region']}  atlas={DATA_INFO['atlas']}  ({DATA_INFO['dataset_dir']})")

CHECKPOINT_SEARCH_DIRS = [
    str(model_root / 'notebooks' / 'checkpoints' / 'checkpoints_gaae_whole_brain'),
]
if OUTPUT_DIR is None:
    OUTPUT_DIR = str(model_root / 'notebooks' / 'checkpoints' / 'checkpoints_longitudinal_common')
os.makedirs(OUTPUT_DIR, exist_ok=True)

# GAAE encoder config (must match the checkpoint).
CONFIG_PATH = model_root / 'configs' / 'gaae_delcode_whole_brain.json'

# Training hyperparameters come entirely from the runner (RESOLVED_CONFIG = dataclass
# defaults < config_path json < hyperparams). Standalone use without a config falls
# back to each adapter's typed defaults.
TRAIN_CONFIG = dict(RESOLVED_CONFIG) if RESOLVED_CONFIG else {}
TRAIN_CONFIG.setdefault('cohort', cohort_tag)
N_FOLDS = int(TRAIN_CONFIG.get('n_folds', 5))
print('Config set.  TRAIN_CONFIG keys:', sorted(TRAIN_CONFIG))

# Defer the in-domain/external test reads (DOCS/temporal-first-ablation.md's
# 2026-08-24 "Evaluation & Comparison Protocol" addendum §4: each is read
# exactly once, after the ladder is frozen — never during a ladder run). A
# ladder arm sets 'defer_test_eval: true' in its hyperparams; the Test-Set /
# External Test-Set / ROC / Early-Detection / Trajectory cells below then all
# no-op, and the one frozen read happens later via common.frozen_read from the
# comparison notebook, reusing this run's saved checkpoint + OOF threshold.
DEFER_TEST_EVAL = bool(TRAIN_CONFIG.get('defer_test_eval', False))
if DEFER_TEST_EVAL:
    print('defer_test_eval=True: in-domain/external test cells below will be skipped.')

# ── External test cohort (optional) ─────────────────────────────────────
# TRAIN_CONFIG['external_test_cohort'] (e.g. 'oasis3') scores ALL of that
# cohort's own subjects once, at the in-domain OOF-derived threshold, after
# the in-domain test-set cell below. None for every existing (non-pooled)
# registry entry, so this is a no-op there. See DOCS/temporal-first-ablation.md
# ("External test: all 60 OASIS-3 subjects, scored once per arm...").
#
# The external cohort is never trained on in this protocol, so its own
# train/val/test split boundary is meaningless here — pooling only its
# 'test.csv' would silently cut external n from 60 to 13 (OASIS-3's own
# 35+12+13 split). Every subject across all three of its downstream splits is
# combined into one external-eval CSV instead. Written atomically (tmp file +
# os.replace) at a content-stable path so the several concurrent ladder runs
# that each resolve this same combined file never observe a partial write —
# every writer produces byte-identical content, so a redundant rewrite is
# harmless, only a genuinely partial one would not be.
EXTERNAL_COHORT = TRAIN_CONFIG.get('external_test_cohort')
EXTERNAL_TEST_CSV = None
if EXTERNAL_COHORT is not None:
    ext_metadata_dir = str(repo_root / 'DATA' / EXTERNAL_COHORT.upper() / '__metadata__')
    ext_splits_dir = os.path.join(ext_metadata_dir, 'SPLITS', 'downstream')
    _ext_parts = [pd.read_csv(os.path.join(ext_splits_dir, f'{s}.csv')) for s in ('train', 'val', 'test')]
    _ext_combined = pd.concat(_ext_parts, ignore_index=True)
    EXTERNAL_TEST_CSV = os.path.join(ext_splits_dir, 'external_all.csv')
    _ext_tmp_path = f'{EXTERNAL_TEST_CSV}.tmp.{os.getpid()}'
    _ext_combined.to_csv(_ext_tmp_path, index=False)
    os.replace(_ext_tmp_path, EXTERNAL_TEST_CSV)
    print(f"External test cohort: {EXTERNAL_COHORT}  ({EXTERNAL_TEST_CSV}, n={len(_ext_combined)})")


Input data: cohort=pooled region=wholebrain  atlas=sch200  (__fc_wholebrain_sch200_flat__)
Config set.  TRAIN_CONFIG keys: ['adjacency_k', 'batch_size', 'beta_kl', 'beta_warmup_epochs', 'change_mask_kappa', 'cohort', 'cohort_conditioning', 'defer_test_eval', 'dim_filter', 'dual_score', 'early_stopping_patience', 'encoder_grad', 'encoder_init', 'epochs', 'external_test_cohort', 'fixed_threshold', 'free_bits', 'fusion', 'gate_rho', 'grad_clip', 'graph_pool', 'gvae_ckpt_path', 'gvae_dropout', 'gvae_heads', 'gvae_hidden', 'gvae_latent', 'lambda_cent', 'lambda_drift', 'lambda_recon', 'lambda_sparse', 'lr', 'lr_factor', 'lr_min', 'lr_patience', 'lstm_dropout', 'lstm_hidden', 'lstm_layers', 'min_visits', 'n_rois', 'node_lstm_ckpt_path', 'node_lstm_init', 'readout', 'recon_target', 'seed', 'shuffle_order', 'shuffle_rng', 'tau', 'threshold_mode', 'use_gate', 'use_scheduler', 'use_time_delta', 'weight_decay', 'zero_time_delta']
defer_test_eval=True: in-domain/external test cells below will be sk

In [8]:
# split-hygiene audit — hard-fails if any subject crosses splits.
_ = run_full_audit({'train': TRAIN_CSV, 'val': VAL_CSV, 'test': TEST_CSV})

# External test cohort must not overlap the CV pool or the in-domain test set —
# a real cross-cohort leak would silently inflate the external score. Reuses
# the same pairwise-disjoint audit above, with the external CSV folded in as a
# fourth named split (its id column, e.g. subject_id, differs from the
# in-domain splits' — run_full_audit auto-detects per CSV).
if EXTERNAL_TEST_CSV is not None:
    _ = run_full_audit({
        'train': TRAIN_CSV, 'val': VAL_CSV, 'test': TEST_CSV, f'external_{EXTERNAL_COHORT}': EXTERNAL_TEST_CSV,
    })
    print(f"[SANITY] External cohort ({EXTERNAL_COHORT}) disjoint from CV pool + in-domain test: OK")


[SANITY] Split sizes: {'train': 187, 'val': 61, 'test': 64}
[SANITY] Pairwise-disjoint: OK


[SANITY] Split sizes: {'train': 187, 'val': 61, 'test': 64, 'external_oasis3': 60}
[SANITY] Pairwise-disjoint: OK
[SANITY] External cohort (oasis3) disjoint from CV pool + in-domain test: OK


## Select GAAE Checkpoint

In [9]:
# Shared helper: resolves GAAE_CHECKPOINT_PATH headlessly under the runner, or
# prompts for an index interactively.
enc_init = str(TRAIN_CONFIG.get('encoder_init', '')).lower()
adapter_tag = str(ADAPTER or MODEL or '').lower()
needs_gaae = not (enc_init == 'none' or adapter_tag == 'braintokengt')
if needs_gaae and GAAE_CHECKPOINT_PATH is None and RUN_DIR is not None:
    raise ValueError(
        "GAAE_CHECKPOINT_PATH is required under the experiment runner. "
        "Set 'checkpoint_path:' on this entry in the experiments/ directory."
    )
if GAAE_CHECKPOINT_PATH is not None or needs_gaae:
    GAAE_RUN_NAME, GAAE_CKPT_PATH, GAAE_RUN_DIR = select_gaae_checkpoint(
        CHECKPOINT_SEARCH_DIRS, checkpoint_path=GAAE_CHECKPOINT_PATH,
    )
    GAAE_CKPT_PATH = str(GAAE_CKPT_PATH)
    print(f'Selected GAAE: {GAAE_RUN_NAME}')
else:
    GAAE_RUN_NAME, GAAE_CKPT_PATH, GAAE_RUN_DIR = 'none', None, None
    print('GAAE checkpoint not required (end-to-end / raw FC baseline).')


GAAE checkpoint not required (end-to-end / raw FC baseline).


In [10]:
if CONFIG_PATH.exists():
    with open(CONFIG_PATH) as f: hp = json.load(f)
    print('GAAE config:', hp)
else:
    hp = dict(latent_dim=64, hidden_dim=128, num_heads=2, cond_dim=2, dropout=0.3,
              adjacency_k=8, file_variant='z_transformed')
    print('GAAE config not found — using defaults.')

GAAE config: {'seed': 100, 'batch_size': 64, 'learning_rate': 0.001, 'weight_decay': 0.001, 'adj_loss_weight': 0.2, 'epochs': 500, 'early_stopping_patience': 25, 'latent_dim': 64, 'hidden_dim': 128, 'num_heads': 2, 'cond_dim': 2, 'dropout': 0.3, 'adjacency_k': 16, 'num_workers': 8, 'file_variant': 'z_transformed'}


## Model adapter

`get_adapter(ADAPTER or MODEL)` resolves the per-model adapter (instantiated with the
GAAE checkpoint, GAAE config, merged training config, and data paths). Its six bound
methods are aliased to the contract-hook names the SHARED cells call.

In [11]:
adapter = get_adapter(ADAPTER or MODEL)(
    gaae_ckpt_path=GAAE_CKPT_PATH, gaae_hp=hp, train_config=TRAIN_CONFIG,
    data_root=WB_DATA_ROOT, cohorts_csv=COHORTS_CSV, device=device, rng=rng,
)
print(f'Adapter: {type(adapter).__name__}  (key={ADAPTER or MODEL})  model_tag={adapter.model_tag}')

# Bind the six contract hooks so every SHARED cell below stays model-agnostic.
build_model          = adapter.build_model
prepare_data         = adapter.prepare_data
train_fold           = adapter.train_fold
eval_split           = adapter.eval_split
truncate_to_n_visits = adapter.truncate_to_n_visits
per_visit_probs      = adapter.per_visit_probs

Adapter: TFGNAdapter  (key=tfgn)  model_tag=tfgn


In [12]:
train_df = pd.read_csv(TRAIN_CSV)
val_df   = pd.read_csv(VAL_CSV)
test_df  = pd.read_csv(TEST_CSV)

# CV pool = train + val; test held out.
cv_pool_df = pd.concat([train_df, val_df], ignore_index=True)
diag_col = 'diagnosis' if 'diagnosis' in cv_pool_df.columns else ('label' if 'label' in cv_pool_df.columns else 'converter_status')
print('CV pool:', cv_pool_df[diag_col].value_counts().to_dict())
print('Test:   ', test_df[diag_col].value_counts().to_dict())


CV pool: {0: 159, 1: 89}
Test:    {0: 40, 1: 24}


In [13]:
# HOOK calls: encode each split into a Bundle (CV pool first — it locks the model's
# input width for GEC), then smoke-test the model.
CV_BUNDLE   = prepare_data(cv_pool_df)
TEST_BUNDLE = prepare_data(test_df)
print(f'CV subjects: {len(CV_BUNDLE)}  Test subjects: {len(TEST_BUNDLE)}')

_ = build_model()   # smoke-test arch construction

LongitudinalSubjectDataset[v2][adni]: 153 subjects (52 converter, 101 stable/MCI)
  min_visits=2; dropped (too few visits)=0
  Scans per subject: min=2  max=10  mean=3.1


LongitudinalSubjectDataset[v2][delcode]: 95 subjects (37 converter, 58 stable/MCI)
  min_visits=2; dropped (too few visits)=0
  Scans per subject: min=2  max=6  mean=3.3


LongitudinalSubjectDataset[v2][adni]: 39 subjects (13 converter, 26 stable/MCI)
  min_visits=2; dropped (too few visits)=0
  Scans per subject: min=2  max=8  mean=3.2


LongitudinalSubjectDataset[v2][delcode]: 25 subjects (11 converter, 14 stable/MCI)
  min_visits=2; dropped (too few visits)=0
  Scans per subject: min=2  max=5  mean=3.0


CV subjects: 248  Test subjects: 64


TFGN model built: trainable=293,057  total=293,057  node_lstm_init=pretrained_finetuned  use_gate=False  recon_target=delta_a_topk  fusion=z_only  readout=mean


## 5-Fold Stratified Subject-Level Cross-Validation

In [14]:
# Open the W&B run, then run the shared CV loop driven by the train_fold hook.
_wb_exp = {'id': EXPERIMENT_ID or 'longitudinal-common', 'mode': MODE or 'longitudinal',
           'model': MODEL or 'GELSTM', 'dataset': DATASET or REGION,
           'seed': SEED, 'wandb': WANDB_ENABLED}
WANDB_RUN = tracking.init_run(_wb_exp, {**(RESOLVED_CONFIG or {}), 'REGION': REGION, 'RUN_NAME': RUN_NAME,
                                        'adapter': ADAPTER or MODEL})

# Out-of-fold, baseline-scan-only (N=1) probe for the Tier-1 static-baseline
# floor (DOCS/temporal-first-ablation.md 2026-08-24 addendum §1: "does
# longitudinal information beat baseline-scan-only?"). Reuses this fold's own
# truncate_to_n_visits/eval_split hooks at this fold's own model + threshold —
# no extra training, no test-set access.
def _static_n1_probe(bundle_va, fold_out):
    n1_bundle = truncate_to_n_visits(bundle_va, 1)
    if len(n1_bundle) == 0:
        return {}
    res = eval_split(fold_out['state_dict'], n1_bundle, fold_out['best_threshold'], device=device)
    return {'prob_n1': dict(zip(res['subject_ids'], res['probs']))}

CV = run_kfold_cv(
    CV_BUNDLE, train_fold, TRAIN_CONFIG,
    n_folds=N_FOLDS, rng=rng, device=device,
    log_fn=lambda d: tracking.log_metrics(WANDB_RUN, d),
    fold_probe=_static_n1_probe,
)

CV_RESULTS             = CV.cv_results
OOF_PROBS, OOF_TARGETS = CV.oof_probs, CV.oof_targets
OOF_SIDS               = CV.oof_sids
OOF_FOLDS              = CV.oof_folds
OOF_EXTRAS             = CV.oof_extras
BEST_MODEL_STATE       = CV.best_model_state
BEST_FOLD              = CV.best_fold
BEST_VAL_AUC           = CV.best_val_auc

Fold 1/5  train=198  val=50


  AUC=0.4670  sens=1.000  spec=0.125  F1=0.562
Fold 2/5  train=198  val=50


  AUC=0.4844  sens=1.000  spec=0.125  F1=0.562
Fold 3/5  train=198  val=50


  AUC=0.6042  sens=0.556  spec=0.719  F1=0.541
Fold 4/5  train=199  val=49


  AUC=0.3805  sens=1.000  spec=0.219  F1=0.576
Fold 5/5  train=199  val=49


  AUC=0.6720  sens=0.722  spec=0.645  F1=0.619



Best fold: 5  CV AUC=0.6720
Youden thr=0.5175  OOF-F1 thr=0.4447


## Cross-Validation Summary

In [15]:
summarize_cv(CV_RESULTS)

Cross-Validation Summary:
Metric                     Mean        Std        Min        Max
------------------------------------------------------------
val_auc                  0.5216     0.1037     0.3805     0.6720
val_sensitivity          0.8556     0.1846     0.5556     1.0000
val_specificity          0.3665     0.2608     0.1250     0.7188
val_f1                   0.5722     0.0261     0.5405     0.6190


In [16]:
# Resolve the active threshold from pooled out-of-fold predictions (Best-F1 default).
# Validation/OOF only — never the test set (see .claude/rules/evaluation.md).
ACTIVE_THRESHOLD, THRESHOLD_METHOD = select_oof_threshold(
    OOF_TARGETS, OOF_PROBS,
    threshold_mode=THRESHOLD_MODE, fixed_threshold=FIXED_THRESHOLD,
    runner_active=RUN_DIR is not None,
)

OOF threshold options:
  [1] Best-F1 (default) thr=0.4447  sens=1.000  spec=0.025  F1=0.535
  [2] Youden            thr=0.5175  sens=0.225  spec=0.843  F1=0.299
Using oof_f1 threshold: 0.4447


## Save Best Model

In [17]:
# Adapter supplies the model-specific descriptors; the shared save_run does the rest.
MODEL_CONFIG = adapter.model_config()
SOURCE_FILES = adapter.source_files()
DATASET_INFO = {**DATA_INFO, 'train_csv': TRAIN_CSV, 'val_csv': VAL_CSV,
                'test_csv': TEST_CSV, 'n_folds': N_FOLDS}

RUN_NAME, RUN_DIR = save_run(
    output_dir=OUTPUT_DIR, run_dir=RUN_DIR, run_name=RUN_NAME,
    model_state=adapter.model_state_for_save(BEST_MODEL_STATE),
    model_config=MODEL_CONFIG, training_config=TRAIN_CONFIG,
    data_info=DATA_INFO, dataset_info=DATASET_INFO, rng=rng,
    best_val_auc=BEST_VAL_AUC, active_threshold=ACTIVE_THRESHOLD, threshold_method=THRESHOLD_METHOD,
    best_fold=BEST_FOLD, cv_results=CV_RESULTS,
    gaae_checkpoint=GAAE_CKPT_PATH, gaae_run_name=GAAE_RUN_NAME,
    source_files=SOURCE_FILES, n_folds=N_FOLDS, model_tag=adapter.model_tag,
)

# Model-specific side artifacts (e.g. dim_filter.npy / scaler.pkl for the GEC-MLP)
# so the comparison/dashboard loaders keep working.
adapter.extra_artifacts(RUN_DIR, BEST_MODEL_STATE)
print(f'Saved to {RUN_DIR}')

try:
    tracking.log_metrics(WANDB_RUN, {'cv_best_val_auc': float(BEST_VAL_AUC),
                                     'active_threshold': float(ACTIVE_THRESHOLD)})
except NameError:
    pass

Saved to /mnt/e/fyassine/ad-early-detection/CLASSIFIER/outputs/tfgn-s1c-recon-pooled-seed45/runs/kind-haze-3-d0d4f8a68-2026-08-24_13-55-50


## Out-of-Fold Evaluation Artifacts (Tiers 1-3)

`DOCS/temporal-first-ablation.md`'s 2026-08-24 "Evaluation & Comparison Protocol"
addendum: every Tier 1-3 number the comparison notebook reads (the stopping
rule, the floor gates, the robustness vetoes) comes from here — computed
purely from `CV_BUNDLE`'s out-of-fold predictions, never from `TEST_BUNDLE` /
`EXTERNAL_BUNDLE`. Writes `oof_predictions.csv` and patches `run_summary.json`'s
`oof` block.

In [18]:
from common.oof import build_oof_frame, oof_metrics
from common.run_artifacts import record_oof_artifacts

OOF_FRAME = build_oof_frame(
    CV_BUNDLE, OOF_SIDS, OOF_PROBS, OOF_TARGETS, OOF_FOLDS, OOF_EXTRAS,
    default_cohort=cohort_tag,
)
OOF_METRICS = oof_metrics(OOF_FRAME, threshold=ACTIVE_THRESHOLD)
record_oof_artifacts(RUN_DIR, OOF_FRAME, OOF_METRICS)

print('OOF metrics (Tier 1-3, computed only from out-of-fold predictions):')
for k, v in OOF_METRICS.items():
    print(f'  {k}: {v}')

try:
    tracking.log_metrics(WANDB_RUN, {f'oof_{k}': v for k, v in OOF_METRICS.items()
                                     if isinstance(v, (int, float))})
except NameError:
    pass

OOF metrics (Tier 1-3, computed only from out-of-fold predictions):
  oof_n: 248
  oof_auc: 0.5076673026641227
  oof_pr_auc: 0.37294106289075657
  oof_balanced_accuracy: 0.5125786163522013
  oof_threshold: 0.44474726915359497
  oof_auc_adni: 0.503046458492003
  oof_auc_delcode: 0.5
  oof_static_n1_auc: 0.5162179351282595
  oof_prob_nscans_spearman_overall: -0.07547705916193903
  oof_prob_nscans_spearman_converter: -0.12423302475169369
  oof_prob_nscans_spearman_non_converter: -0.04583736684956741


In [19]:
import sys as _sys
if '/mnt/e/fyassine/ad-early-detection' not in _sys.path:
    _sys.path.insert(0, '/mnt/e/fyassine/ad-early-detection')
from SHARED.plotting import add_note, format_model_runs_note

MODEL_RUNS = {"GAAE": GAAE_RUN_NAME, adapter.model_tag.upper(): RUN_NAME}


In [20]:
# Post-hoc probability calibration (temperature scaling) fit on the OOF predictions
# (proper held-out — never the test set). Monotonic: AUC/threshold decisions unchanged;
# it widens the probability spread and lowers ECE. Saved as a run artifact so analyses
# (e.g. the visit-count confound sanity notebook) and dashboards can apply it.
from common.calibration import fit_temperature, apply_temperature, expected_calibration_error

TEMPERATURE = fit_temperature(OOF_PROBS, OOF_TARGETS)
ECE_OOF_RAW = expected_calibration_error(OOF_PROBS, OOF_TARGETS)
ECE_OOF_CAL = expected_calibration_error(apply_temperature(OOF_PROBS, TEMPERATURE), OOF_TARGETS)
CALIBRATION = {"temperature": float(TEMPERATURE),
               "ece_oof_raw": float(ECE_OOF_RAW), "ece_oof_cal": float(ECE_OOF_CAL)}
with open(Path(RUN_DIR) / "calibration.json", "w") as f:
    json.dump(CALIBRATION, f, indent=2)
print(f"Temperature={TEMPERATURE:.3f}   OOF ECE {ECE_OOF_RAW:.3f} -> {ECE_OOF_CAL:.3f}")

try:
    tracking.log_metrics(WANDB_RUN, {"temperature": float(TEMPERATURE),
                                     "ece_oof_raw": float(ECE_OOF_RAW),
                                     "ece_oof_cal": float(ECE_OOF_CAL)})
except NameError:
    pass

Temperature=0.584   OOF ECE 0.133 -> 0.128


## Cohort-Decoding Probe (pooled runs only)

`DOCS/temporal-first-ablation.md` — "Cohort-shift control": a pooled ADNI+DELCODE
model could learn cohort identity as a shortcut instead of disease signal. This
probe decodes cohort from each CV-pool subject's patient embedding with a 5-fold
logistic regression. No-op unless the adapter implements the optional
`patient_embeddings` hook (TFGN does; GELSTM/GEC/BrainTokenGT do not, so this
silently skips for every existing registry entry) **and** the run is pooled.

In [21]:
# Uses the winning fold's model (BEST_MODEL_STATE) to embed the full CV pool —
# not strict per-fold OOF (train_fold's return contract doesn't carry
# per-fold embeddings), but sufficient for this diagnostic's purpose (catching
# a model that has trivially learned cohort identity), and consistent with how
# BEST_MODEL_STATE is already reused for TEST_BUNDLE / trajectory scoring below.
COHORT_PROBE_AUC = None
if hasattr(adapter, 'patient_embeddings') and any('cohort' in it for it in CV_BUNDLE.items):
    from sklearn.linear_model import LogisticRegression
    from sklearn.model_selection import cross_val_score
    from SHARED.provenance import patch_run_summary

    cohort_labels = [it['cohort'] for it in CV_BUNDLE.items]
    unique_cohorts = sorted(set(cohort_labels))
    if len(unique_cohorts) == 2:
        EMB = adapter.patient_embeddings(BEST_MODEL_STATE, CV_BUNDLE, device=device)
        y_cohort = np.array([unique_cohorts.index(c) for c in cohort_labels])
        probe_aucs = cross_val_score(
            LogisticRegression(max_iter=1000), EMB, y_cohort, cv=5, scoring='roc_auc',
        )
        COHORT_PROBE_AUC = float(np.mean(probe_aucs))
        print(f"Cohort probe AUC ({unique_cohorts[0]} vs {unique_cohorts[1]}): "
              f"{COHORT_PROBE_AUC:.4f}  (5-fold CV on patient_embeddings; >0.75 on the "
              f"winning arm triggers the escalation rule in DOCS/temporal-first-ablation.md)")
        patch_run_summary(RUN_DIR, {'cohort_probe_auc': COHORT_PROBE_AUC})
    else:
        print(f"Cohort probe skipped: expected exactly 2 cohorts, found {unique_cohorts}")
else:
    print("Cohort probe skipped (single-cohort run, or adapter has no patient_embeddings hook).")

Cohort probe AUC (adni vs delcode): 0.8138  (5-fold CV on patient_embeddings; >0.75 on the winning arm triggers the escalation rule in DOCS/temporal-first-ablation.md)


## Test-Set Evaluation

In [22]:
# HOOK: score the held-out test set at the val-derived threshold (no test leakage).
# Skipped when defer_test_eval=True — DOCS/temporal-first-ablation.md's 2026-08-24
# addendum §4: the in-domain test set is read exactly once, after the ladder is
# frozen, via common.frozen_read from the comparison notebook.
TEST_METRICS = None
if not DEFER_TEST_EVAL:
    TEST_METRICS = eval_split(BEST_MODEL_STATE, TEST_BUNDLE, ACTIVE_THRESHOLD, device=device)

    print('Test-Set Results')
    print('=' * 40)
    print(f"AUC:         {TEST_METRICS['auc']:.4f}")
    print(f"Sensitivity: {TEST_METRICS['sensitivity']:.4f}")
    print(f"Specificity: {TEST_METRICS['specificity']:.4f}")
    print(f"F1:          {TEST_METRICS['f1']:.4f}")
    print(f"Threshold:   {ACTIVE_THRESHOLD:.4f}  ({THRESHOLD_METHOD})")
    print()
    print(classification_report(TEST_METRICS['targets'],
                                (np.asarray(TEST_METRICS['probs']) >= ACTIVE_THRESHOLD).astype(int),
                                target_names=['stable_mci', 'converter']))

    record_test_metrics(RUN_DIR, TEST_METRICS, threshold=ACTIVE_THRESHOLD, threshold_method=THRESHOLD_METHOD)
    print(f"Test metrics saved to {RUN_DIR / 'run_summary.json'}")

    try:
        tracking.log_metrics(WANDB_RUN, {'test_auc': float(TEST_METRICS['auc']), 'test_f1': float(TEST_METRICS['f1']),
                                         'test_sensitivity': float(TEST_METRICS['sensitivity']),
                                         'test_specificity': float(TEST_METRICS['specificity'])})
    except NameError:
        pass
else:
    print('Test-Set Evaluation skipped (defer_test_eval=True).')

Test-Set Evaluation skipped (defer_test_eval=True).


## External Test-Set Evaluation (optional)

Scores `TRAIN_CONFIG['external_test_cohort']`'s own downstream test split — e.g.
OASIS-3 for a pooled ADNI+DELCODE run — at the same in-domain OOF-derived
threshold used above. No-op (skipped) unless `EXTERNAL_TEST_CSV` was resolved in
the Configuration cell. See `DOCS/temporal-first-ablation.md`: this score is
descriptive only and never used to select an arm.

In [23]:
from common.run_artifacts import record_external_metrics

EXTERNAL_METRICS = None
if not DEFER_TEST_EVAL and EXTERNAL_TEST_CSV is not None:
    ext_df = pd.read_csv(EXTERNAL_TEST_CSV)
    # A native single-cohort CSV (e.g. OASIS-3's own test.csv) has no 'cohort'
    # column; tag it so prepare_data routes through build_multicohort_bundle
    # and resolves EXTERNAL_COHORT's own FC root (COHORT_ROOTS), not
    # WB_DATA_ROOT (which for a pooled run points at the ADNI+DELCODE-only
    # symlink farm and would silently find zero external files).
    if 'cohort' not in ext_df.columns:
        ext_df['cohort'] = EXTERNAL_COHORT

    # Same HOOK path as the in-domain test: prepare_data -> eval_split, at the
    # already-fixed ACTIVE_THRESHOLD (never re-optimized on this cohort).
    EXTERNAL_BUNDLE = prepare_data(ext_df)
    EXTERNAL_METRICS = eval_split(BEST_MODEL_STATE, EXTERNAL_BUNDLE, ACTIVE_THRESHOLD, device=device)

    print(f'External Test-Set Results ({EXTERNAL_COHORT})')
    print('=' * 40)
    print(f"AUC:         {EXTERNAL_METRICS['auc']:.4f}")
    print(f"Sensitivity: {EXTERNAL_METRICS['sensitivity']:.4f}")
    print(f"Specificity: {EXTERNAL_METRICS['specificity']:.4f}")
    print(f"F1:          {EXTERNAL_METRICS['f1']:.4f}")
    print(f"n subjects:  {len(EXTERNAL_BUNDLE)}")

    record_external_metrics(
        RUN_DIR, EXTERNAL_METRICS, threshold=ACTIVE_THRESHOLD,
        threshold_method=THRESHOLD_METHOD, cohort=EXTERNAL_COHORT,
    )
    print(f"External metrics saved to {RUN_DIR / 'run_summary.json'} (ext_{EXTERNAL_COHORT}_auc)")

    try:
        tracking.log_metrics(WANDB_RUN, {f'ext_{EXTERNAL_COHORT}_auc': float(EXTERNAL_METRICS['auc']),
                                         f'ext_{EXTERNAL_COHORT}_f1': float(EXTERNAL_METRICS['f1'])})
    except NameError:
        pass
elif DEFER_TEST_EVAL and EXTERNAL_TEST_CSV is not None:
    print('External Test-Set Evaluation skipped (defer_test_eval=True).')

External Test-Set Evaluation skipped (defer_test_eval=True).


## ROC Curves

In [24]:
if not DEFER_TEST_EVAL:
    fig = plot_oof_test_roc(
        OOF_TARGETS, OOF_PROBS, TEST_METRICS['targets'], TEST_METRICS['probs'],
        title=f'{type(adapter).__name__}  |  GAAE: {GAAE_RUN_NAME}',
    )
    add_note(fig, list(fig.axes), format_model_runs_note(MODEL_RUNS))
    plt.show()
else:
    print('ROC Curves skipped (defer_test_eval=True; no TEST_METRICS to plot against).')

ROC Curves skipped (defer_test_eval=True; no TEST_METRICS to plot against).


## Early-Detection Curve: AUC vs. Number of Visits Used

Shared `early_detection_table`: for each N it restricts every test subject with `T >= N`
visits to their first N (via `truncate_to_n_visits`) and re-scores with `eval_split`.
Rows with fewer than 4 subjects or a single class are skipped. `N=1` = single-scan.

In [25]:
ED_ROWS = []
if not DEFER_TEST_EVAL:
    ED_ROWS = early_detection_table(
        TEST_BUNDLE, eval_split, truncate_to_n_visits,
        BEST_MODEL_STATE, ACTIVE_THRESHOLD, device=device,
    )
else:
    print('Early-Detection Curve skipped (defer_test_eval=True).')

Early-Detection Curve skipped (defer_test_eval=True).


## Conversion Probability Trajectories

Shared `trajectory_frame` builds per-visit `P(converter)` for each test subject with
>= 2 visits (via `per_visit_probs`); `plot_conversion_trajectories` renders the panels.

In [26]:
if not DEFER_TEST_EVAL:
    TRAJ_DF = trajectory_frame(TEST_BUNDLE, per_visit_probs, BEST_MODEL_STATE, device=device)
    print(f"Trajectory data: {TRAJ_DF['pid'].nunique()} subjects")
    fig = plot_conversion_trajectories(
        TRAJ_DF, ACTIVE_THRESHOLD,
        title=f'Per-visit conversion probability trajectories  |  {type(adapter).__name__}',
    )
    add_note(fig, list(fig.axes), format_model_runs_note(MODEL_RUNS))
    plt.show()
else:
    print('Conversion Probability Trajectories skipped (defer_test_eval=True).')

# Always finalize the run's W&B session, whether or not the test/external
# cells above executed (DEFER_TEST_EVAL skips them, but the run itself is
# still complete at this point — see the Configuration cell).
try:
    tracking.finish_run(WANDB_RUN)
except NameError:
    pass

Conversion Probability Trajectories skipped (defer_test_eval=True).
